# Improving Model

In [1]:
import pickle

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Cargar features
with open("featuresVGG.pkl", "rb") as f:
    features = pickle.load(f)

# Cargar descripciones
with open("descriptions.pkl", "rb") as f:
    descriptions = pickle.load(f)

vocab_size = len(tokenizer.word_index) + 1

def max_length(descriptions):
    return max(len(d.split()) for desc in descriptions.values() for d in desc)

max_len = max_length(descriptions)
print("Vocab size:", vocab_size)
print("Max caption length:", max_len)


from tensorflow.keras.utils import pad_sequences, to_categorical
import numpy as np

def data_generator(descriptions, features, tokenizer, max_len, batch_size):
    while True:
        X1, X2, y = [], [], []
        n = 0
        for key, desc_list in descriptions.items():
            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_len)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

                    X1.append(features[key])
                    X2.append(in_seq)
                    y.append(out_seq)

                    n += 1
                    if n == batch_size:
                        yield ([np.array(X1), np.array(X2)], np.array(y))
                        X1, X2, y = [], [], []
                        n = 0


Vocab size: 8574
Max caption length: 34


In [2]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def generate_caption(model, tokenizer, photo, max_len):
    in_text = 'startseq'
    for _ in range(max_len):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_len)
        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = tokenizer.index_word.get(yhat)
        if word is None:
            break
        in_text += ' ' + word
        if word == 'endseq':
            break
    return in_text.replace('startseq ', '').replace(' endseq', '')

### Create new models with Hyperparameters

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.optimizers import Adam

def create_model(vocab_size, max_len, learning_rate):
    # Imagen
    inputs1 = Input(shape=(4096,))
    fe1 = Dropout(0.4)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    # Texto
    inputs2 = Input(shape=(max_len,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = Dropout(0.4)(se1)
    se3 = LSTM(256)(se2)

    # Fusionar
    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)

    # Compilar con optimizador ajustable
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='categorical_crossentropy', optimizer=optimizer)

    return model


##### This creates new models with different epochs, batch sizes and learning rates

In [ ]:
import time
import itertools
import os  

# Seleccionar parametros a probar
epoch_options = [30]
batch_size_options = [64]
learning_rate_options = [1e-3]

# Todas las combinaciones posibles
param_grid = list(itertools.product(epoch_options, batch_size_options, learning_rate_options))

for idx, (epochs, batch_size, learning_rate) in enumerate(param_grid):
    model_name = f"caption_model_e{epochs}_b{batch_size}_lr{learning_rate:.0e}.h5"
    
    if os.path.exists(model_name):
        print(f"\n⏩ Modelo ya existe: {model_name}, saltando...")
        continue

    print(f"\n🔁 Combinación {idx+1}/{len(param_grid)}")
    print(f"Epochs: {epochs}, Batch Size: {batch_size}, Learning Rate: {learning_rate}")

    # Crear modelo con hiperparámetros actuales
    model = create_model(vocab_size, max_len, learning_rate)

    steps = sum(len(c) for c in descriptions.values()) // batch_size
    start_time = time.time()

    for epoch in range(epochs):
        print(f"  Epoch {epoch+1}/{epochs}")
        generator = data_generator(descriptions, features, tokenizer, max_len, batch_size)
        model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)

    # Guardar el modelo entrenado
    model.save(model_name)

    elapsed_time = time.time() - start_time
    print(f"✅ Guardado: {model_name}")   
    print(f"⏱️ Tiempo total: {elapsed_time:.2f} segundos")
    print("-----------------------------------------------------")



🔁 Combinación 1/1
Epochs: 30, Batch Size: 64, Learning Rate: 0.001
  Epoch 1/30
632/632 [==============================] - 280s 438ms/step - loss: 5.6813
  Epoch 2/30
632/632 [==============================] - 295s 467ms/step - loss: 4.7048
  Epoch 3/30
632/632 [==============================] - 330s 522ms/step - loss: 4.2467
  Epoch 4/30
632/632 [==============================] - 319s 505ms/step - loss: 3.9453
  Epoch 5/30
632/632 [==============================] - 329s 520ms/step - loss: 3.7002
  Epoch 6/30
632/632 [==============================] - 330s 522ms/step - loss: 3.4975
  Epoch 7/30
632/632 [==============================] - 326s 516ms/step - loss: 3.2986
  Epoch 8/30
632/632 [==============================] - 332s 525ms/step - loss: 3.1318
  Epoch 9/30
632/632 [==============================] - 334s 528ms/step - loss: 2.9737
  Epoch 10/30
632/632 [==============================] - 335s 530ms/step - loss: 2.8282
  Epoch 11/30
632/632 [==============================] - 338s

c:\Users\WD\.conda\envs\tf_gpu\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


✅ Guardado: caption_model_e30_b64_lr1e-03.h5
⏱️ Tiempo total: 10407.78 segundos
-----------------------------------------------------


In [ ]:
from tensorflow.keras.models import load_model
model_hiper = load_model(".h5")

# Verificar el resumen del modelo
model_hiper.summary()

### Evaluating models

In [ ]:
import os
from tensorflow.keras.models import load_model
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm

# Parámetros usados en el entrenamiento
epoch_options = [30]
batch_size_options = [64]
learning_rate_options = [1e-3, 5e-5]

# Ruta de los modelos (ajusta si es necesario)
model_dir = "./"

# Lista para guardar resultados
results = []

# Iterar sobre todas las combinaciones posibles
for epochs in epoch_options:
    for batch_size in batch_size_options:
        for lr in learning_rate_options:
            # Construir nombre de archivo
            model_filename = f"caption_model_e{epochs}_b{batch_size}_lr{lr:.0e}.h5"
            model_path = os.path.join(model_dir, model_filename)

            # Verificar si el archivo existe
            if os.path.exists(model_path):
                print(f"\n📂 Evaluando modelo: {model_filename}")
                model = load_model(model_path)

                # Evaluación del BLEU Score
                actual, predicted = [], []
                test_keys = list(features.keys())[:1000]

                for key in tqdm(test_keys):
                    # Referencias reales
                    references = [d.split() for d in descriptions[key]]
                    photo = features[key].reshape((1, 4096))
                    y_pred = generate_caption(model, tokenizer, photo, max_len).split()
                    actual.append(references)
                    predicted.append(y_pred)

                # Calcular BLEU
                bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
                bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
                bleu3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0))
                bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))

                # Guardar resultados
                results.append({
                    'model': model_filename,
                    'epochs': epochs,
                    'batch_size': batch_size,
                    'learning_rate': lr,
                    'BLEU-1': bleu1,
                    'BLEU-2': bleu2,
                    'BLEU-3': bleu3,
                    'BLEU-4': bleu4
                })

                print(f"✅ BLEU Scores:")
                print(f"   BLEU-1: {bleu1:.4f}")
                print(f"   BLEU-2: {bleu2:.4f}")
                print(f"   BLEU-3: {bleu3:.4f}")
                print(f"   BLEU-4: {bleu4:.4f}")
            else:
                print(f"❌ Modelo no encontrado: {model_filename}")

# Mostrar resumen ordenado por BLEU-4 descendente
results = sorted(results, key=lambda x: x['BLEU-4'], reverse=True)
print("\n📊 Resultados ordenados por BLEU-4:")
for res in results:
    print(f"{res['model']}")
    print(f"   BLEU-1: {res['BLEU-1']:.4f}")
    print(f"   BLEU-2: {res['BLEU-2']:.4f}")
    print(f"   BLEU-3: {res['BLEU-3']:.4f}")
    print(f"   BLEU-4: {res['BLEU-4']:.4f}\n")



📂 Evaluando modelo: caption_model_e30_b64_lr1e-03.h5


100%|██████████| 1000/1000 [09:51<00:00,  1.69it/s]


✅ BLEU Scores:
   BLEU-1: 0.4111
   BLEU-2: 0.2695
   BLEU-3: 0.1859
   BLEU-4: 0.1292
❌ Modelo no encontrado: caption_model_e30_b64_lr5e-05.h5

📊 Resultados ordenados por BLEU-4:
caption_model_e30_b64_lr1e-03.h5
   BLEU-1: 0.4111
   BLEU-2: 0.2695
   BLEU-3: 0.1859
   BLEU-4: 0.1292

